# Development vs Production Split

**Goal:** split the 32 annotated notes into 16 for development (model + prompt selection) and 16 for production (final, one-shot evaluation of the winning combination).

**Design:** the sample was built as 16 strata (8 clusters × 2 length categories) with exactly 2 notes each. Assigning **1 note per stratum to each split** guarantees identical cluster and length representation on both sides — a perfectly balanced stratified split, where the only randomness is *which* of the 2 notes of each stratum goes to which side (fixed seed).

The split is frozen to disk so every downstream script and notebook reads the exact same assignment.

In [ ]:
import os
import pandas as pd

from clinical_notes_extraction.config import PROJECT_ROOT

In [ ]:

# Constants local to this notebook
SAMPLE_FILE = f"{PROJECT_ROOT}/scripts/3_information_extraction/3_2_medications_on_admission/data/sample.parquet"  # the 32 annotated notes
OUTPUT_DIR = f'{PROJECT_ROOT}/scripts/3_information_extraction/3_2_medications_on_admission/data/sample' # the 32 annotated notes
RANDOM_SEED = 42

# Group by Columns
COLS = ['cluster', 'meds_on_admission_cleaned_length_binary']

## Load and sanity-check the sample

We expect exactly 32 notes and exactly 2 notes in each of the 16 strata.

In [ ]:
df_sample = pd.read_parquet(SAMPLE_FILE)

assert len(df_sample) == 32, f"Expected 32 notes, got {len(df_sample)}"

strata_counts = df_sample.groupby(COLS).size()
assert len(strata_counts) == 16, f"Expected 16 strata, got {len(strata_counts)}"
assert (strata_counts == 2).all(), "Every stratum must contain exactly 2 notes"

print(f"{len(df_sample)} notes across {len(strata_counts)} strata — OK")

## Split: one note per stratum to each side

Within each stratum the 2 notes are shuffled with a fixed seed; the first goes to dev, the second to prod.

In [ ]:
dev_rows, prod_rows = [], []

for (cluster, length), group in df_sample.groupby(COLS):
    shuffled = group.sample(frac=1, random_state=RANDOM_SEED)
    dev_rows.append(shuffled.iloc[0])
    prod_rows.append(shuffled.iloc[1])

dev = pd.DataFrame(dev_rows).reset_index(drop=True)
prod = pd.DataFrame(prod_rows).reset_index(drop=True)

print(f"dev: {len(dev)} notes | prod: {len(prod)} notes")

## Verify representation

Both splits must show the exact same cluster × length distribution (1 note per cell). ICU status is reported post-hoc only — mild imbalance is acceptable and simply documented.

In [ ]:
dev

In [ ]:
prod

In [ ]:
print("DEV — cluster x length:")
print(pd.crosstab(dev[COLS[0]], dev[COLS[1]]))
print("\PROD — cluster x length:")
print(pd.crosstab(prod[COLS[0]], prod[COLS[1]]))

In [ ]:
# Guard against patient-level overlap between splits. One-note-per-patient
# sampling makes this impossible by construction, but the assert is cheap
# and documents the guarantee explicitly.
if "subject_id" in df_sample.columns:
    overlap = set(dev["subject_id"]) & set(prod["subject_id"])
    assert not overlap, f"Patient leakage between splits: {overlap}"
    print("No patient overlap between splits — OK")

## Freeze the split to disk

These two files are the single source of truth for all downstream steps (`run_extraction.py` and both evaluation notebooks).

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
dev.to_parquet(f'{OUTPUT_DIR}/dev_sample.parquet', index=False)
prod.to_parquet(f'{OUTPUT_DIR}/prod_sample.parquet', index=False)
print(f"Saved: {OUTPUT_DIR}/dev_sample.parquet and {OUTPUT_DIR}/prod_sample.parquet")